### Прогоны тестов на добавление товаров со стороны модуля <font style="color:green;"><b><i>prestashop</i></b></font>

# <a id='toc1_'></a>[Amazon тест](#toc0_)

## `Murano Glass` - Прогоняю весь сценарий на добавление товаров со страницы категории
## Для корркетной работы этого сценария я переопределяю локаторы для товаров из категории по url `test_url_4`
есть проблема с локаторами для разных типов товара: `new` и `ref`
в частности есть проблема с нахождением цены. 

In [ ]:
from pathlib import Path
from typing import Union
import header
from header import gs, \
            logger,  logs_and_errors_decorator,  jprint, pprint, \
            SN, SF, \
            Product, ProductFields, \
            Supplier, start_supplier


supplier_prefix = 'amazon'
s = start_supplier(supplier_prefix)
""" s - на протяжении всего кода означает класс `Supplier` """

print(" Можно продолжать ")


from dict_scenarios import scenario
s.current_scenario = scenario['Murano Glass']

# -------------- сокращения -------
l = s.reread_locators('product')
d = s.driver
_ = d.execute_locator
# --------------------------------


In [ ]:

test_url_4 = r"https://www.amazon.com/C%C3%A1-dOro-Hippie-Colored-Murano-Style/dp/B09N53XSQB/ref=sr_1_1_sspa?crid=24Q0ZZYVNOQMP&keywords=Art+Deco+murano+glass&qid=1687277030&sprefix=art+deco+murano+glass%2Caps%2C230&sr=8-1-spons&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&psc=1"
test_url_5 = r"https://www.amazon.com/Luxury-Lane-Sommerso-Centerpiece-Decoration/dp/B0BSZBF8NJ/ref=sr_1_3_sspa?c=ts&keywords=Vases&qid=1688326048&s=furniture&sr=1-3-spons&ts_id=3745451&sp_csd=d2lkZ2V0TmFtZT1zcF9hdGY&th=1"
d.get_url(test_url_4)

_______________________________________________
## <a id='toc1_3_'></a>[Поля товара](#toc0_)


`grab_product_page` <p>Это функция из модуля <font color="11073B"> suppliers.amazon.grabber</font>
Ее можно тестировать. 
<h6>Не забудь!</h6>
<font color='red'><b>Глявное - не забыть потом синхронизирoвать в suppliers.amazon.grabber</b><font></p>

### Проверяю наличие товара в бд

In [ ]:
l = s.reread_locators('product')
ASIN = _(l['ASIN'])

link_rewrite = d.current_url.split(f'''/''')[-4]

l = s.reread_locators('product')
print(l['link_rewrite'])

product_reference = f"{s.supplier_id}-{ASIN}"
response = Product.check_if_product_in_presta_db(product_reference)
pprint(f' Если товар в бд получу id_product, иначе False. Получил: {response}')

In [ ]:

def grab_product_page(s: Supplier, id_product: int = 0 ) -> ProductFields:
    """ ПОКА ТОЛЬКО ДЛЯ НОВЫХ!!!!! СТРАНИЦ
        Собираю информацию со страницы товара. 
        Важно помнить, что драйвер уже должен быть на
        этой странице
        ---------------
        Attrs:
            s (Supplier)
        @returns
            f (ProductFields) с заполненными полями, else False
    """


    l = s.reread_locators('product')
    d = s.driver
   
    _ = d.execute_locator

    print('Strat local function grabber')

    def add_new_product_stage_1(s: Supplier) -> ProductFields:
        """ Первая стадия добавления нового товара. Заполняю все необходимые поля
            Далее я отправлю их в prestashop и получу обратно ID вновь созданного товара
            На второй стадии, зная ID, я отправлю главную картинку и прочее
        """
        f: ProductFields = ProductFields()
       
        """ ID,ASIN,SKU,SUPPLIER SKU """
    

        def _set_defaults() -> bool:
            """ Set defaults for product of supplier """
            f.active = 1
            f.on_sale = 1
            f.min_qty = 1
            f.low_stock_level = 0
            f.low_stock_threshold = ''
            f.show_price = 1
            f.show_condition = 1
            f.aviable_online_only = 0
            f.advanced_stock_management = 0
            f.state = 1

        def set_price(s, format: str = 'str') -> Union[str,float]:
            """ Привожу денюшку к 
            [ ] float 
            [v] str
            """
            l = s.reread_locators('product')
            raw_price = _(l['price']['new'])[0]
            raw_price = str(raw_price).split('\n')[0]
            return SN.normalize_price(raw_price)
    

        """ Я добавляю в базу данных престашоп товар путем нескольких последовательных действий
        1. Добавляю поля, необходимые для создания нового товара
        2. Получаю `id_product` созданного товара
        3. Используя полученный `id_product` загружаю дефолтную картинку
        4. итд.
        """

        _set_defaults()
        l = s.reread_locators('product')
        ASIN = _(l['ASIN'])
        f.reference = f'{s.supplier_id}-{ASIN}'
        f.supplier_reference = ASIN
        f.price = set_price(s, format = 'str')
        f.name = _(l['name'])[0]
        f.summary = _(l['summary'])[0]
        f.id_supplier = s.supplier_id
        
        _category_default = list(s.current_scenario['presta_categories']['default_category'].keys())[0]
        f.id_category_default = _category_default
        f.category_ids = Product.get_parent_categories(_category_default)

        affiliate = _(l['affiliate short link'])
        affiliate = affiliate[1][0]
        f.affiliate_short_link = affiliate

        f.link_rewrite = d.current_url.split(f'''/''')[-4]

        #f.link_rewrite = "test-link"
        return f

    def fields_4_new_product_stage_2(s: Supplier, f: ProductFields) -> ProductFields:
        """ После того, как я занес новый товар в бд - я получу его id
        Далее я гружу фото и получаю ее id
        Далее я догружаю осталные па
        ----------------------
        Attrs: 
            s (Supplier):
            f (ProductFields): Заполненные на первом этапе поля. Я беру из них только то, что мне надо для апдейта
        @returns
            ProductFields: поля для апдейта
        """
        l = s.reread_locators('product')
        image_url = _(l['Images URLs (x,y,z...)'])[0]
        img = Product.upload_product_default_image(f.id_product, image_url)

        category_ids = Product.get_parent_categories(f.id_category_default)
        Product.update_product_categories_in_prestashop(category_ids)
        return f

    
    
    if id_product == 0:
        f = add_new_product_stage_1(s)
        product = Product.add_new_product_to_prestashop(f.product_dict)
        print("---------------------NEW PRODUCT-----------------------")
        pprint(product)
        id_product = product['id']
        print("---------------------NEW PRODUCT ID-----------------------")
        print(id_product)
    
    f_stage2 = ProductFields()
    f_stage2.id_product = id_product
    
    f_stage2.id_category_default = list(s.current_scenario['presta_categories']['default_category'].keys())[0]
    print("-------------- FIELDS 2------------------------------")
    pprint(f_stage2.fields)
    f_stage2 = fields_4_new_product_stage_2(s, f_stage2)

    prod_dict: dict = {'product': f_stage2.fields}
    #prod_dict: dict = f_stage2.fields
    print("--------------------------------------------")
    pprint(prod_dict)
    print("--------------------------------------------")
    jprint(prod_dict)
    print("--------------------------------------------")


    prod = Product.update_product_in_prestashop(id_product, prod_dict)
    pass


In [ ]:

""" Манипуляции с товаром """

if not isinstance(response, bool):
    """ Если не вернулся False, значит товар уже в бд, я получу его id_product
    здесь обработка product_update
    """
    id_product = response
    image_url = _(l['Images URLs (x,y,z...)'])[0]
    print(image_url)
    Product.upload_product_default_image(id_product, image_url)
    pass

else:
    
    #родная функция
    #product_fields: ProductFields = Product.grab_product_page(s)

    # местная функция
    #product_fields: ProductFields = grab_product_page(s)
    
    pass


In [ ]:
product_fields: ProductFields = Product.grab_product_page(s)
#product_fields: ProductFields = grab_product_page(s)

In [ ]:
jprint(product_fields.fields)

In [ ]:
product_dict: dict = {}
product_dict['product'] = product_fields.fields

In [ ]:
jprint(product_dict)

In [ ]:
#prod = PrestaAPIV1.add('products', product_dict)

In [ ]:
pprint(prod['product'])

# Погоняй до сюда

In [ ]:
l = s.reread_locators('product')
affiliate = _(l['affiliate short link'])
pprint(l['affiliate short link'])
pprint(affiliate)

In [ ]:

product_dict['product']['link_rewrite']: dict = \
{
    "language":[
        {'attrs': {'id': '1'},'value': link_rewrite},
        {'attrs': {'id': '1'},'value': link_rewrite},
        {'attrs': {'id': '1'},'value': link_rewrite}
]}


product_dict['product']['affiliate_short_link']: dict = \
{
    "language":[
        {'attrs': {'id': '1'},'value': affiliate[1][0]},
        {'attrs': {'id': '1'},'value': affiliate[1][0]},
        {'attrs': {'id': '1'},'value': affiliate[1][0]}
]}



pprint(product_dict)

In [ ]:
prod = PrestaAPIV1OdAhad.add('products', product_dict)

In [ ]:
pprint(PrestaAPIV1OdAhad.add('products', product_dict))
#pprint(Product.add_new_product_to_prestashop(product_dict))

In [ ]:
PrestaAPIV1.add('products', product_dict)

In [ ]:
test_p = Product.get_product_info_by_id(63)